# Capacity simulation

Runs **`capacity_simulation.py`** as a subprocess with `cwd = ok1/` so imports, `./data/`, and CLI flags match the README.

**CUDA / geomstats:** pass `--device cuda` in flags; `recall_config.early_set_geomstats_backend_from_argv()` reads `argv` inside the subprocess. Batched Karcher on CPU applies to synthetic and `--device cpu` image paths.

### Setup
- Select the repo venv as the Jupyter kernel (`ok1/.venv`).
- Prefer starting Jupyter **from `ok1/`** or **`ok1/notebooks/`** so the resolver below finds `capacity_simulation.py`.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    """Locate ok1 directory (directory that contains capacity_simulation.py)."""
    here = Path.cwd().resolve()
    candidates = [here, *here.parents[:4]]
    for d in candidates:
        if (d / "capacity_simulation.py").is_file():
            return d
    raise RuntimeError(
        "Could not find capacity_simulation.py. cd to ok1/ or ok1/notebooks/, "
        "or open the notebook so cwd is inside the repo."
    )


REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def run_capacity(*argv: str) -> None:
    script = REPO_ROOT / "capacity_simulation.py"
    cmd = [sys.executable, str(script), *argv]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

### Fast sanity: synthetic dataset

Writes under `outputs/dim{d}/Radius{mem-R}/result.csv` and `recall_plot.png` (see `icml_hyp.config.recall_config.synthetic_csv_plot_paths`).

In [ ]:
run_capacity(
    "--dataset",
    "synthetic",
    "--d",
    "10",
    "--M-min",
    "25",
    "--M-max",
    "120",
    "--n-trials",
    "2",
    "--max-steps",
    "8",
    "--device",
    "cpu",
    "--noise_sigma",
    "0.35",
)

### Optional: MNIST/CIFAR (heavier)

Uncomment and adjust `--device cuda` if you want GPU Euclidean models; image runs mirror README patterns.

In [ ]:
# run_capacity(
#     "--dataset", "mnist",
#     "--pca-dim", "50",
#     "--M-min", "10",
#     "--M-max", "200",
#     "--n-trials", "2",
#     "--max-steps", "5",
#     "--mem-R", "3",
#     "--device", "cpu",
# )
#
# run_capacity(
#     "--dataset", "cifar10",
#     "--no-pca",
#     "--M-min", "10",
#     "--M-max", "60",
#     "--n-trials", "1",
#     "--max-steps", "5",
#     "--mem-R", "2",
#     "--device", "cpu",
# )

### Inspect last synthetic outputs

Adjust `--d` and `--mem-R` to match your run if needed.

In [ ]:
from argparse import Namespace

import pandas as pd
from IPython.display import display
from icml_hyp.config.recall_config import synthetic_csv_plot_paths

args = Namespace(d=10, output_dir="outputs", mem_R=2)
csv_path, plot_path = synthetic_csv_plot_paths(args)

full_csv = REPO_ROOT / csv_path
full_png = REPO_ROOT / plot_path
print("CSV:", full_csv.resolve())
print("Plot:", full_png.resolve())
if full_csv.is_file():
    display(pd.read_csv(full_csv).head(12))